In [114]:
from pathlib import Path
import zipfile

import pandas as pd
import searoute as sr
from geopy.distance import great_circle
import coordinates as coord
import os
from core_modules import core_modules as core

In this code, I assemble the necessary data, clean it, conduct an exploratory data analysis, and then ...

# Data Collection

## Grid intensity

Grid intensity is important for manufacturing, but it's still relevant for supply chains. Source: [Ember Energy](https://ember-energy.org/latest-insights/global-electricity-review-2025/major-countries-and-regions/)



In [115]:
# gCO2/kWh - grid intensity is about CO2 released per unit of energy. Mostly about manufacturing, but still relevant
grid_intensity = {"china": 525, "mexico": 412, "s_korea": 390}


## Emission Factor
Ton-km emission factor is CO2 released per ton of commodity per kilometer transported. Relevant for transportation, depends on country's mix of transportation methods used. Because washing machines are transported mainly by trade vessels and trucks, these two are the main methods used in the calculation of the emission factor.

Lane-specific emission factors combine the IMO Fourth GHG Study global average with adjustments for typical vessel deployment on each lane (sourced from UNCTAD 2024 Chapter II) and feeder-megaship transshipment patterns documented in Notteboom & Rodrigue (2009).

Instead of country names, country codes were used as per [country.txt](https://www.census.gov/foreign-trade/schedules/c/country.txt) file on Census.gov.

Sources:
+ [US EPA SmartWay Carrier Emission Factors](epa.gov/smartway) - emission factor of Mexico-US trade routes
+ [New shipping routes highlight growing Asia-to-Mexico trade](https://www.freightwaves.com/news/new-shipping-routes-highlight-growing-asia-to-mexico-trade) - emission factor of Asia-Pacific trade routes
+ [Review of Maritime Transport 2024: Navigating Maritime Chokepoints](https://unctad.org/publication/review-maritime-transport-2024.) - GHG global average with country-specific adjustments for each trade route.
+ [Notteboom and Rodrigue](https://doi.org/10.1007/s10708-008-9210-4) - documents shipment patterns of feeder megaships

In [116]:
# gCO2/ton-km  - emission factor is CO2 released per ton of commodity per kilometer transported. Different countries use different methods

# ton_km = {
#     "china": 4,
#     "south_korea": 4.5, 
#     "vietnam": 7, 
#     "india": 6,
#     "mexico": 80 
# }

# ton_km = {
#     "5700": 4,
#     "5800": 4.5, 
#     "5520": 7, 
#     "5330": 6,
#     "2010": 80 
# }

## Distance

This code calculates distances between ports for countries beyond the ocean(China, India, South Korea, Vietnam), as well as land distance over the US border for Mexico.

Because the ISTHS6M and PORTHS6MM is encoded through fixed-width text, it is difficult to read and convert into dataframes. To solve this, I read it once, then saved the snapshot into Parquet. That way

### ISTHS6M (December 2024 — Pre-Tariff)

In [117]:
_snapshot = Path('data/snapshots/ISTHSM2412.parquet')
if _snapshot.exists():
    df_land_imports_2412 = pd.read_parquet(_snapshot)
else:
    with zipfile.ZipFile('data/original/ISTHSM2412.ZIP') as zf:
        with zf.open(zf.namelist()[0]) as f:
            df_land_imports_2412 = pd.read_fwf(
                f,
                colspecs=coord.colspecs_land,
                names=coord.names_land,
                dtype={c: str for c in coord.str_cols_land},
            )
    _snapshot.parent.mkdir(exist_ok=True)
    df_land_imports_2412.to_parquet(_snapshot, index=False)
df_land_imports_2412.head()


,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr
0,010121,1220,AK,2024,12,0,0,0,0,0,...,0,0,20337,20337,0,0,0,0,0,0
1,010121,1220,AR,2024,12,0,0,0,0,0,...,0,0,7500,7500,0,0,0,0,0,0
2,010121,1220,CA,2024,12,0,0,0,0,0,...,0,0,3632,3632,0,0,0,0,0,0
3,010121,1220,DE,2024,12,0,0,0,0,0,...,0,0,2183,2183,0,0,0,0,0,0
4,010121,1220,FL,2024,12,138066,138066,0,0,0,...,0,0,453343,453343,0,0,0,0,0,0


In [118]:
df_land_imports_2412.info()

<class 'pandas.DataFrame'>
RangeIndex: 1194681 entries, 0 to 1194680
Data columns (total 21 columns):
 #   Column      Non-Null Count    Dtype
---  ------      --------------    -----
 0   commodity   1194681 non-null  str  
 1   cty_code    1194681 non-null  str  
 2   state       1194681 non-null  str  
 3   year        1194681 non-null  str  
 4   month       1194681 non-null  str  
 5   gen_val_mo  1194681 non-null  int64
 6   con_val_mo  1194681 non-null  int64
 7   air_val_mo  1194681 non-null  int64
 8   air_swt_mo  1194681 non-null  int64
 9   ves_val_mo  1194681 non-null  int64
 10  ves_swt_mo  1194681 non-null  int64
 11  cnt_val_mo  1194681 non-null  int64
 12  cnt_swt_mo  1194681 non-null  int64
 13  gen_val_yr  1194681 non-null  int64
 14  con_val_yr  1194681 non-null  int64
 15  air_val_yr  1194681 non-null  int64
 16  air_swt_yr  1194681 non-null  int64
 17  ves_val_yr  1194681 non-null  int64
 18  ves_swt_yr  1194681 non-null  int64
 19  cnt_val_yr  1194681 non-null  in

### ISTHS6M (December 2025 — Post-Tariff)

In [119]:
_snapshot = Path('data/snapshots/ISTHSM2512.parquet')
if _snapshot.exists():
    df_land_imports_2512 = pd.read_parquet(_snapshot)
else:
    with zipfile.ZipFile('data/original/ISTHSM2512.ZIP') as zf:
        with zf.open(zf.namelist()[0]) as f:
            df_land_imports_2512 = pd.read_fwf(
                f,
                colspecs=coord.colspecs_land,
                names=coord.names_land,
                dtype={c: str for c in coord.str_cols_land},
            )
    _snapshot.parent.mkdir(exist_ok=True)
    df_land_imports_2512.to_parquet(_snapshot, index=False)
df_land_imports_2512.head()


,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr
0,010121,1220,CA,2025,12,25000,25000,0,0,0,...,0,0,25000,25000,0,0,0,0,0,0
1,010121,1220,CO,2025,12,6000,6000,0,0,0,...,0,0,12000,12000,0,0,0,0,0,0
2,010121,1220,CT,2025,12,0,0,0,0,0,...,0,0,3237,3237,0,0,0,0,0,0
3,010121,1220,FL,2025,12,164429,164429,0,0,0,...,0,0,251214,251214,0,0,0,0,0,0
4,010121,1220,ID,2025,12,0,0,0,0,0,...,0,0,22000,22000,0,0,0,0,0,0


This is the code that could be used to auto-import data. However, the problem with it is that there is poor connection between the API and the website.œ

In [120]:
# import requests
# from pathlib import Path

# OUT = Path("data/port_hs6")
# OUT.mkdir(parents=True, exist_ok=True)

# def url(year, month):
#     yy = str(year)[2:]
#     return (f"https://www.census.gov/trade/downloads/{year}"
#             f"/Port/im_hs6_m/PORTHS6MM{yy}{month:02d}.ZIP")

# # Your tariff treatment window
# for year in (2023, 2024, 2025):
#     for month in range(1, 13):
#         u = url(year, month)
#         out_path = OUT / f"PORTHS6MM{str(year)[2:]}{month:02d}.ZIP"
#         if out_path.exists():
#             continue
#         r = requests.get(u, timeout=30)
#         if r.status_code == 200:
#             out_path.write_bytes(r.content)
#             print(f"saved {out_path}")
#         else:
#             print(f"missing or not yet released: {u}")

## Weight of imports

Similar to UN Comtrade, but focuses on the USA and is more laconic.

[USA Trade Census](https://usatrade.census.gov/data/Perspective60/Browse/browsetables.aspx?utosid=5687ae9fc5be588295a68da15cd2f1cd&cache=tffv5e)
+ Data Source Selection: State Import Data(Harmonized System)
+ Filters:
    + Measures: Vessel SWT and Air SWT(kg) - The gross weight in kilograms of shipments made by seafaring vessel/airplane at customs
        + No data on land transportation there
    + State: All States
    + Commodity: 845011, 845012, 845019 and 845020(washing machines)
    + Country: India, South Korea, Mexico
    + Time: Jan 2025 - Mar 2026(monthly)

In [121]:
washing_machine_trade_filepath = "data/original/State Imports by HS Commodities_v4.csv"
washing_machine_df = pd.read_csv(washing_machine_trade_filepath, index_col=False, header=2)

In [122]:
washing_machine_df.head()

,Commodity,Country,Time,Air SWT (kg),Vessel SWT (kg),Unnamed: 5
0,845011 Washing Mach Automatic W Dry Line Cap N...,China,January 2025,174,"1,777,528",NaN
1,845011 Washing Mach Automatic W Dry Line Cap N...,China,February 2025,NaN,"2,459,827",NaN
2,845011 Washing Mach Automatic W Dry Line Cap N...,China,March 2025,NaN,"2,268,490",NaN
3,845011 Washing Mach Automatic W Dry Line Cap N...,China,April 2025,NaN,"2,512,291",NaN
4,845011 Washing Mach Automatic W Dry Line Cap N...,China,May 2025,NaN,"2,541,012",NaN


# Data Cleaning
Cleaning is the longest and the most important step of any data cycle. After all, without good data there can be no good results. Because this projects uses data from a wide variety of sources, this means dealing with many different formats, which may complicate the cleaning process even further.

In [123]:
# this defines the columns in the PORTHS6MM dataframe, which are to be converted to numbers
numerical_cols_asian = ["gen_val_mo", "air_val_mo", "air_swt_mo", "ves_val_mo", "ves_swt_mo", "cnt_val_mo", "cnt_swt_mo"]


## Ocean-based Distance/Weight Data (Asian Routes, Pre-Tariff — Dec 2024)

In [124]:
_snapshot = Path('data/snapshots/PORTHS6MM2412.parquet')
if _snapshot.exists():
    df_ocean_routes_2412 = pd.read_parquet(_snapshot)
else:
    with zipfile.ZipFile('data/original/PORTHS6MM2412.ZIP') as zf:
        with zf.open(zf.namelist()[0]) as f:
            df_ocean_routes_2412 = pd.read_fwf(
                f,
                colspecs=coord.colspecs_sea,
                names=coord.names_sea,
                dtype={c: str for c in coord.str_cols_sea},
            )
    _snapshot.parent.mkdir(exist_ok=True)
    df_ocean_routes_2412.to_parquet(_snapshot, index=False)


In [125]:
df_ocean_routes_2412.info()

<class 'pandas.DataFrame'>
RangeIndex: 1212596 entries, 0 to 1212595
Data columns (total 20 columns):
 #   Column       Non-Null Count    Dtype
---  ------       --------------    -----
 0   commodity    1212596 non-null  str  
 1   cty_code     1212596 non-null  str  
 2   dist_unlade  1212596 non-null  str  
 3   port_unlade  1212596 non-null  str  
 4   year         1212596 non-null  str  
 5   month        1212596 non-null  str  
 6   gen_val_mo   1212596 non-null  int64
 7   air_val_mo   1212596 non-null  int64
 8   air_swt_mo   1212596 non-null  int64
 9   ves_val_mo   1212596 non-null  int64
 10  ves_swt_mo   1212596 non-null  int64
 11  cnt_val_mo   1212596 non-null  int64
 12  cnt_swt_mo   1212596 non-null  int64
 13  gen_val_yr   1212596 non-null  int64
 14  air_val_yr   1212596 non-null  int64
 15  air_swt_yr   1212596 non-null  int64
 16  ves_val_yr   1212596 non-null  int64
 17  ves_swt_yr   1212596 non-null  int64
 18  cnt_val_yr   1212596 non-null  int64
 19  cnt_swt_yr 

In [126]:
df_ocean_routes_2412["port_full"] = df_ocean_routes_2412["dist_unlade"] + df_ocean_routes_2412["port_unlade"]

df_asian_2412 = df_ocean_routes_2412[df_ocean_routes_2412["cty_code"].isin(coord.asian_origins)].T.drop_duplicates().T
for col in numerical_cols_asian:
    df_asian_2412[col] = pd.to_numeric(df_asian_2412[col])

df_asian_wash_2412 = df_asian_2412[df_asian_2412["commodity"].isin(["845011", "845020"])].copy()
df_asian_wash_2412 = df_asian_wash_2412[df_asian_wash_2412["port_full"].isin(coord.us_ports)]
df_asian_wash_2412 = df_asian_wash_2412[df_asian_wash_2412["ves_swt_mo"] > 0]
df_asian_wash_2412["distance"] = df_asian_wash_2412.apply(
    core.compute_distance, args=(coord.asian_origins, coord.us_ports, True, "cty_code", "port_full"), axis=1)
df_asian_wash_2412["co2"] = df_asian_wash_2412.apply(core.compute_co2, args=["ves_swt_mo"], axis=1)

df_asian_wash_2412.to_csv("data/intermediate/PORTHS6MM_asian_wash_2412.csv")
df_asian_wash_2412.info()

<class 'pandas.DataFrame'>
Index: 130 entries, 799729 to 800031
Data columns (total 23 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   commodity    130 non-null    object 
 1   cty_code     130 non-null    object 
 2   dist_unlade  130 non-null    object 
 3   port_unlade  130 non-null    object 
 4   year         130 non-null    object 
 5   month        130 non-null    object 
 6   gen_val_mo   130 non-null    int64  
 7   air_val_mo   130 non-null    int64  
 8   air_swt_mo   130 non-null    int64  
 9   ves_val_mo   130 non-null    int64  
 10  ves_swt_mo   130 non-null    int64  
 11  cnt_val_mo   130 non-null    int64  
 12  cnt_swt_mo   130 non-null    int64  
 13  gen_val_yr   130 non-null    object 
 14  air_val_yr   130 non-null    object 
 15  air_swt_yr   130 non-null    object 
 16  ves_val_yr   130 non-null    object 
 17  ves_swt_yr   130 non-null    object 
 18  cnt_val_yr   130 non-null    object 
 19  cnt_swt_yr   130

In [127]:
df_asian_wash_2412["port_full"].value_counts().sort_index(ascending=False)

port_full
5301    6
5201    4
4909    4
4110    1
4103    2
4102    1
4101    2
3901    3
3801    3
3604    1
3501    1
3401    1
3310    1
3126    2
3004    2
3002    6
3001    6
2904    5
2811    6
2809    1
2801    1
2720    1
2709    6
2704    6
2006    3
2002    2
1901    5
1803    5
1801    6
1703    6
1601    6
1501    1
1401    6
1303    6
1012    1
1003    6
1001    3
0901    1
0401    1
Name: count, dtype: int64

## Ocean-based Distance/Weight Data (Asian Routes, Post-Tariff — Dec 2025)

In [128]:
_snapshot = Path("data/snapshots/PORTHS6MM2512.parquet")
if _snapshot.exists():
    df_ocean_routes_2512 = pd.read_parquet(_snapshot)
else:
    with zipfile.ZipFile("data/original/PORTHS6MM2512.ZIP") as zf:
        with zf.open(zf.namelist()[0]) as f:
            df_ocean_routes_2512 = pd.read_fwf(
                f,
                colspecs=coord.colspecs_sea,
                names=coord.names_sea,
                dtype={c: str for c in coord.str_cols_sea},
            )
    _snapshot.parent.mkdir(exist_ok=True)
    df_ocean_routes_2512.to_parquet(_snapshot, index=False)

df_ocean_routes_2512["port_full"] = df_ocean_routes_2512["dist_unlade"] + df_ocean_routes_2512["port_unlade"]

df_asian_2512 = df_ocean_routes_2512[df_ocean_routes_2512["cty_code"].isin(coord.asian_origins)].T.drop_duplicates().T
for col in numerical_cols_asian:
    df_asian_2512[col] = pd.to_numeric(df_asian_2512[col])

df_asian_wash_2512 = df_asian_2512[df_asian_2512["commodity"].isin(["845011", "845020"])].copy()
df_asian_wash_2512 = df_asian_wash_2512[df_asian_wash_2512["port_full"].isin(coord.us_ports)]
df_asian_wash_2512 = df_asian_wash_2512[df_asian_wash_2512["ves_swt_mo"] > 0]
df_asian_wash_2512["distance"] = df_asian_wash_2512.apply(
    core.compute_distance, args=(coord.asian_origins, coord.us_ports, True, "cty_code", "port_full"), axis=1)
df_asian_wash_2512["co2"] = df_asian_wash_2512.apply(core.compute_co2, args=["ves_swt_mo"], axis=1)

df_asian_wash_2512.to_csv("data/intermediate/PORTHS6MM_asian_wash_2512.csv")
df_asian_wash_2512.info()

<class 'pandas.DataFrame'>
Index: 122 entries, 824147 to 824435
Data columns (total 23 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   commodity    122 non-null    object 
 1   cty_code     122 non-null    object 
 2   dist_unlade  122 non-null    object 
 3   port_unlade  122 non-null    object 
 4   year         122 non-null    object 
 5   month        122 non-null    object 
 6   gen_val_mo   122 non-null    int64  
 7   air_val_mo   122 non-null    int64  
 8   air_swt_mo   122 non-null    int64  
 9   ves_val_mo   122 non-null    int64  
 10  ves_swt_mo   122 non-null    int64  
 11  cnt_val_mo   122 non-null    int64  
 12  cnt_swt_mo   122 non-null    int64  
 13  gen_val_yr   122 non-null    object 
 14  air_val_yr   122 non-null    object 
 15  air_swt_yr   122 non-null    object 
 16  ves_val_yr   122 non-null    object 
 17  ves_swt_yr   122 non-null    object 
 18  cnt_val_yr   122 non-null    object 
 19  cnt_swt_yr   122

In [129]:
df_asian_wash_2512["port_full"].value_counts()

port_full
1003    6
1303    6
1601    6
1703    6
1801    6
2704    6
2709    6
2811    6
3001    6
3002    6
5301    6
1401    5
5201    5
1803    5
1901    4
4909    4
3901    3
2002    2
3004    2
2904    2
1001    2
3501    2
1108    1
4101    1
4103    1
5206    1
3126    1
2303    1
2304    1
2507    1
1012    1
2006    1
2809    1
3201    1
3604    1
3802    1
4110    1
4115    1
5203    1
0712    1
2720    1
3303    1
Name: count, dtype: int64

## Ocean-based Distance/Weight Data (Asian Routes, Control — HS 8528 TVs, Dec 2024)

In [130]:
df_asian_tv_2412 = df_asian_2412[df_asian_2412["commodity"].str.startswith("8528")].copy()
df_asian_tv_2412 = df_asian_tv_2412[df_asian_tv_2412["port_full"].isin(coord.us_ports)]
df_asian_tv_2412 = df_asian_tv_2412[df_asian_tv_2412["ves_swt_mo"] > 0]
df_asian_tv_2412["distance"] = df_asian_tv_2412.apply(
    core.compute_distance, args=(coord.asian_origins, coord.us_ports, True, "cty_code", "port_full"), axis=1)
df_asian_tv_2412["co2"] = df_asian_tv_2412.apply(core.compute_co2, args=["ves_swt_mo"], axis=1)

df_asian_tv_2412.to_csv("data/intermediate/PORTHS6MM_asian_tv_2412.csv")
df_asian_tv_2412.info()

<class 'pandas.DataFrame'>
Index: 687 entries, 953166 to 956000
Data columns (total 23 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   commodity    687 non-null    object 
 1   cty_code     687 non-null    object 
 2   dist_unlade  687 non-null    object 
 3   port_unlade  687 non-null    object 
 4   year         687 non-null    object 
 5   month        687 non-null    object 
 6   gen_val_mo   687 non-null    int64  
 7   air_val_mo   687 non-null    int64  
 8   air_swt_mo   687 non-null    int64  
 9   ves_val_mo   687 non-null    int64  
 10  ves_swt_mo   687 non-null    int64  
 11  cnt_val_mo   687 non-null    int64  
 12  cnt_swt_mo   687 non-null    int64  
 13  gen_val_yr   687 non-null    object 
 14  air_val_yr   687 non-null    object 
 15  air_swt_yr   687 non-null    object 
 16  ves_val_yr   687 non-null    object 
 17  ves_swt_yr   687 non-null    object 
 18  cnt_val_yr   687 non-null    object 
 19  cnt_swt_yr   687

In [131]:
TvPortcodes2412 = df_asian_tv_2412["port_full"].value_counts().sort_index()
with pd.option_context('display.max_rows', None,
                       'display.max_columns', None,
                       'display.precision', 3,
                       ):
    print(TvPortcodes2412)

port_full
0106     2
0115     1
0212     3
0401     5
0408     1
0417     5
0701     2
0708    10
0712    12
0901    13
1001     6
1002     1
1003    15
1012    19
1101     9
1102     1
1104     3
1108     6
1301     1
1303     6
1401    12
1501     3
1503     2
1512     3
1601     9
1603     3
1703    14
1704    12
1791     4
1801     3
1803     4
1808     6
1816     4
1901     6
1910     1
2002    15
2006     5
2007     1
2301     3
2302     1
2304     5
2305     5
2401     3
2402     3
2404     1
2408     3
2501     4
2505     1
2506     5
2507     2
2604     5
2605     2
2704    19
2707     1
2709    17
2720    19
2721     3
2722     5
2801    15
2809     6
2811    10
2834     3
2904     4
2910     1
3001     9
3002    13
3003     1
3004     8
3009     1
3019     1
3029    10
3105     1
3126    20
3201     3
3205     9
3279     7
3302     1
3303     1
3307     4
3310     2
3401     9
3403     3
3411     1
3422     1
3501     7
3512     1
3604     3
3613     2
3801    12
3802    12


## Ocean-based Distance/Weight Data (Asian Routes, Control — HS 8528 TVs, Dec 2025)

In [132]:
df_asian_tv_2512 = df_asian_2512[df_asian_2512["commodity"].str.startswith("8528")].copy()
df_asian_tv_2512 = df_asian_tv_2512[df_asian_tv_2512["port_full"].isin(coord.us_ports)]
df_asian_tv_2512 = df_asian_tv_2512[df_asian_tv_2512["ves_swt_mo"] > 0]
df_asian_tv_2512["distance"] = df_asian_tv_2512.apply(
    core.compute_distance, args=(coord.asian_origins, coord.us_ports, True, "cty_code", "port_full"), axis=1)
df_asian_tv_2512["co2"] = df_asian_tv_2512.apply(core.compute_co2, args=["ves_swt_mo"], axis=1)

df_asian_tv_2512.to_csv("data/intermediate/PORTHS6MM_asian_tv_2512.csv")
df_asian_tv_2512.info()

<class 'pandas.DataFrame'>
Index: 701 entries, 978446 to 981197
Data columns (total 23 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   commodity    701 non-null    object 
 1   cty_code     701 non-null    object 
 2   dist_unlade  701 non-null    object 
 3   port_unlade  701 non-null    object 
 4   year         701 non-null    object 
 5   month        701 non-null    object 
 6   gen_val_mo   701 non-null    int64  
 7   air_val_mo   701 non-null    int64  
 8   air_swt_mo   701 non-null    int64  
 9   ves_val_mo   701 non-null    int64  
 10  ves_swt_mo   701 non-null    int64  
 11  cnt_val_mo   701 non-null    int64  
 12  cnt_swt_mo   701 non-null    int64  
 13  gen_val_yr   701 non-null    object 
 14  air_val_yr   701 non-null    object 
 15  air_swt_yr   701 non-null    object 
 16  ves_val_yr   701 non-null    object 
 17  ves_swt_yr   701 non-null    object 
 18  cnt_val_yr   701 non-null    object 
 19  cnt_swt_yr   701

In [133]:
df_asian_tv_2512["port_full"].value_counts().sort_index()
TvPortcodes2512 = df_asian_tv_2512["port_full"].value_counts().sort_index()
with pd.option_context('display.max_rows', None,
                       'display.max_columns', None,
                       'display.precision', 3,
                       ):
    print(TvPortcodes2512)

port_full
0106     1
0115     1
0209     1
0212     2
0401     5
0408     2
0417     6
0701     2
0708     6
0712    11
0901    14
1001     9
1002     1
1003    16
1012    15
1101     7
1102     1
1108     6
1109     1
1303     4
1401    10
1501     1
1503     3
1512     4
1601     9
1603     1
1701     1
1703    17
1704    17
1791     6
1801     3
1803     3
1808     3
1809     1
1816     3
1901     8
1910     1
2001     1
2002    17
2006     4
2007     2
2301     4
2302     1
2304     7
2305    11
2401     3
2402     1
2408     2
2501     1
2506     6
2507     2
2604     6
2605     3
2608     1
2704    23
2707     1
2709    18
2720    18
2721     5
2722     7
2801    19
2809     4
2810     1
2811    11
2834     1
2835     1
2904     2
2910     2
3001    10
3002     9
3004     7
3009     1
3020     1
3029    12
3126    24
3201     3
3205    11
3279     8
3303     4
3307     3
3310     4
3401    10
3403     5
3501     8
3502     1
3604     3
3801     8
3802    13
3803     1
3807     8


So, upon filtering and cleaning data, we recognize that the vast majority of imports of Mexican washing machines into the USA is done through the land, rather than sea or plane. This confirmed my initial hypothesis on Mexico's transformation breakdown and now justifies the plan to calculate solely the inland leg's carbon footprint for imports from Mexico.

Now, we need to understand the distances from each state to state.

In [134]:
# import requests
# from pathlib import Path

# OUT = Path("data/port_hs6")
# OUT.mkdir(parents=True, exist_ok=True)

# def url(year, month):
#     yy = str(year)[2:]
#     return (f"https://www.census.gov/trade/downloads/{year}"
#             f"/Port/im_hs6_m/PORTHS6MM{yy}{month:02d}.ZIP")

# # Your tariff treatment window
# for year in (2023, 2024, 2025):
#     for month in range(1, 13):
#         u = url(year, month)
#         out_path = OUT / f"PORTHS6MM{str(year)[2:]}{month:02d}.ZIP"
#         if out_path.exists():
#             continue
#         r = requests.get(u, timeout=30)
#         if r.status_code == 200:
#             out_path.write_bytes(r.content)
#             print(f"saved {out_path}")
#         else:
#             print(f"missing or not yet released: {u}")

## Land-based Distance/Weight Data (Mexico, Pre-Tariff — Dec 2024)

In [135]:
df_land_imports_mex_2412 = df_land_imports_2412[df_land_imports_2412["cty_code"] == "2010"]
df_land_imports_mex_2412.head()

,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr
31,010121,2010,AZ,2024,12,0,0,0,0,0,...,0,0,44862,44862,0,0,0,0,0,0
32,010121,2010,CA,2024,12,0,0,0,0,0,...,0,0,33250,33250,0,0,0,0,0,0
33,010121,2010,FL,2024,12,0,0,0,0,0,...,0,0,29000,29000,29000,3500,0,0,0,0
34,010121,2010,NM,2024,12,0,0,0,0,0,...,0,0,3000,3000,0,0,0,0,0,0
35,010121,2010,TX,2024,12,0,0,0,0,0,...,0,0,232003,232003,0,0,0,0,0,0


In [136]:
df_land_imports_mex_v1_2412 = df_land_imports_mex_2412.T.drop_duplicates().T
df_land_imports_mex_v1_2412.info()

<class 'pandas.DataFrame'>
Index: 39825 entries, 31 to 1194457
Data columns (total 21 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   commodity   39825 non-null  object
 1   cty_code    39825 non-null  object
 2   state       39825 non-null  object
 3   year        39825 non-null  object
 4   month       39825 non-null  object
 5   gen_val_mo  39825 non-null  object
 6   con_val_mo  39825 non-null  object
 7   air_val_mo  39825 non-null  object
 8   air_swt_mo  39825 non-null  object
 9   ves_val_mo  39825 non-null  object
 10  ves_swt_mo  39825 non-null  object
 11  cnt_val_mo  39825 non-null  object
 12  cnt_swt_mo  39825 non-null  object
 13  gen_val_yr  39825 non-null  object
 14  con_val_yr  39825 non-null  object
 15  air_val_yr  39825 non-null  object
 16  air_swt_yr  39825 non-null  object
 17  ves_val_yr  39825 non-null  object
 18  ves_swt_yr  39825 non-null  object
 19  cnt_val_yr  39825 non-null  object
 20  cnt_swt_yr  39825 n

In [137]:
numerical_cols = ["gen_val_mo", "con_val_mo", "air_val_mo", "air_swt_mo", "ves_val_mo", "ves_swt_mo", "cnt_val_mo", "cnt_swt_mo"]

for col in numerical_cols:
    df_land_imports_mex_v1_2412[col] = pd.to_numeric(df_land_imports_mex_v1_2412[col])
df_land_imports_mex_v1_2412.info()

<class 'pandas.DataFrame'>
Index: 39825 entries, 31 to 1194457
Data columns (total 21 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   commodity   39825 non-null  object
 1   cty_code    39825 non-null  object
 2   state       39825 non-null  object
 3   year        39825 non-null  object
 4   month       39825 non-null  object
 5   gen_val_mo  39825 non-null  int64 
 6   con_val_mo  39825 non-null  int64 
 7   air_val_mo  39825 non-null  int64 
 8   air_swt_mo  39825 non-null  int64 
 9   ves_val_mo  39825 non-null  int64 
 10  ves_swt_mo  39825 non-null  int64 
 11  cnt_val_mo  39825 non-null  int64 
 12  cnt_swt_mo  39825 non-null  int64 
 13  gen_val_yr  39825 non-null  object
 14  con_val_yr  39825 non-null  object
 15  air_val_yr  39825 non-null  object
 16  air_swt_yr  39825 non-null  object
 17  ves_val_yr  39825 non-null  object
 18  ves_swt_yr  39825 non-null  object
 19  cnt_val_yr  39825 non-null  object
 20  cnt_swt_yr  39825 n

In [138]:
df_land_imports_mex_wash_2412 = df_land_imports_mex_v1_2412[df_land_imports_mex_v1_2412["commodity"].isin(["845011", "845020"])]
df_land_imports_mex_wash_2412.head()

,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr
799500,845011,2010,CA,2024,12,2805860,2805860,0,0,0,...,0,0,28767168,28767168,0,0,40635,35100,40635,35100
799501,845011,2010,CO,2024,12,430796,430796,0,0,0,...,0,0,5790782,5790782,0,0,0,0,0,0
799502,845011,2010,FL,2024,12,1334984,1334984,0,0,328736,...,328736,73407,19850931,19850931,0,0,3583492,831963,3583492,831963
799503,845011,2010,GA,2024,12,1199977,1199977,0,0,0,...,0,0,16540032,16540032,0,0,0,0,0,0
799504,845011,2010,IN,2024,12,1157949,1157949,0,0,0,...,0,0,13289441,13289441,0,0,0,0,0,0


In [139]:
df_land_imports_mex_wash_2412["state"].value_counts()

state
CA    2
CO    2
FL    2
GA    2
IN    2
KY    2
MD    2
OH    2
PR    2
TX    2
WA    2
NC    1
AZ    1
IL    1
MA    1
MI    1
MN    1
MO    1
NE    1
NJ    1
PA    1
Name: count, dtype: int64

In [140]:
df_land_imports_mex_wash_v1_2412 = df_land_imports_mex_wash_2412.copy()

df_land_imports_mex_wash_v1_2412["distance"] = df_land_imports_mex_wash_v1_2412.apply(core.compute_distance,
    args=[coord.mexico_start, coord.state_centroids, False, None, "state"], axis=1)

Because the other dataset(ISTHS6MM) does not have land transportation weight and only land transportation value, I will calculate average value of washing machines and use it to find land transportation weight for Mexico. Because there are no rows with 0 for vessel-transported value or vessel-transported weight, I do not need to worry about rows with 0s affecting the mean.

In [141]:
washing_machine_price_coeff = df_asian_wash_2412["ves_val_mo"].mean()/df_asian_wash_2412["ves_swt_mo"].mean()

print(washing_machine_price_coeff, "$/kg")

3.8833822563329905 $/kg


In [142]:
df_land_imports_mex_wash_v1_2412["gen_swt_mo"] = df_land_imports_mex_wash_v1_2412["gen_val_mo"] / washing_machine_price_coeff

In [143]:
df_land_imports_mex_wash_v1_2412["co2"] = df_land_imports_mex_wash_v1_2412.apply(core.compute_co2, args=["gen_swt_mo"], axis=1)
df_land_imports_mex_wash_v1_2412.info()

<class 'pandas.DataFrame'>
Index: 32 entries, 799500 to 799719
Data columns (total 24 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   commodity   32 non-null     object 
 1   cty_code    32 non-null     object 
 2   state       32 non-null     object 
 3   year        32 non-null     object 
 4   month       32 non-null     object 
 5   gen_val_mo  32 non-null     int64  
 6   con_val_mo  32 non-null     int64  
 7   air_val_mo  32 non-null     int64  
 8   air_swt_mo  32 non-null     int64  
 9   ves_val_mo  32 non-null     int64  
 10  ves_swt_mo  32 non-null     int64  
 11  cnt_val_mo  32 non-null     int64  
 12  cnt_swt_mo  32 non-null     int64  
 13  gen_val_yr  32 non-null     object 
 14  con_val_yr  32 non-null     object 
 15  air_val_yr  32 non-null     object 
 16  air_swt_yr  32 non-null     object 
 17  ves_val_yr  32 non-null     object 
 18  ves_swt_yr  32 non-null     object 
 19  cnt_val_yr  32 non-null     object 
 2

In [144]:
df_land_imports_mex_wash_v1_2412.to_csv("data/intermediate/ISTHS6MM_mex_2412.csv")

## Land-based Distance/Weight Data (Mexico, Post-Tariff — Dec 2025)

In [145]:
df_land_imports_mex_2512 = df_land_imports_2512[df_land_imports_2512["cty_code"] == "2010"]
df_land_imports_mex_2512.head()

,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr
26,010121,2010,FL,2025,12,24000,24000,24000,2000,0,...,0,0,57000,57000,57000,5500,0,0,0,0
160,010129,2010,AZ,2025,12,0,0,0,0,0,...,0,0,275173,275173,0,0,0,0,0,0
161,010129,2010,CA,2025,12,0,0,0,0,0,...,0,0,504000,504000,504000,4500,0,0,0,0
162,010129,2010,FL,2025,12,51700,51700,51700,4000,0,...,0,0,143200,143200,143200,12957,0,0,0,0
163,010129,2010,NM,2025,12,0,0,0,0,0,...,0,0,195600,195600,0,0,0,0,0,0


In [146]:
df_land_imports_mex_v1_2512 = df_land_imports_mex_2512.T.drop_duplicates().T
df_land_imports_mex_v1_2512.info()

<class 'pandas.DataFrame'>
Index: 39926 entries, 26 to 1247779
Data columns (total 21 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   commodity   39926 non-null  object
 1   cty_code    39926 non-null  object
 2   state       39926 non-null  object
 3   year        39926 non-null  object
 4   month       39926 non-null  object
 5   gen_val_mo  39926 non-null  object
 6   con_val_mo  39926 non-null  object
 7   air_val_mo  39926 non-null  object
 8   air_swt_mo  39926 non-null  object
 9   ves_val_mo  39926 non-null  object
 10  ves_swt_mo  39926 non-null  object
 11  cnt_val_mo  39926 non-null  object
 12  cnt_swt_mo  39926 non-null  object
 13  gen_val_yr  39926 non-null  object
 14  con_val_yr  39926 non-null  object
 15  air_val_yr  39926 non-null  object
 16  air_swt_yr  39926 non-null  object
 17  ves_val_yr  39926 non-null  object
 18  ves_swt_yr  39926 non-null  object
 19  cnt_val_yr  39926 non-null  object
 20  cnt_swt_yr  39926 n

In [147]:
numerical_cols = ["gen_val_mo", "con_val_mo", "air_val_mo", "air_swt_mo", "ves_val_mo", "ves_swt_mo", "cnt_val_mo", "cnt_swt_mo"]

for col in numerical_cols:
    df_land_imports_mex_v1_2512[col] = pd.to_numeric(df_land_imports_mex_v1_2512[col])
df_land_imports_mex_v1_2512.info()

<class 'pandas.DataFrame'>
Index: 39926 entries, 26 to 1247779
Data columns (total 21 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   commodity   39926 non-null  object
 1   cty_code    39926 non-null  object
 2   state       39926 non-null  object
 3   year        39926 non-null  object
 4   month       39926 non-null  object
 5   gen_val_mo  39926 non-null  int64 
 6   con_val_mo  39926 non-null  int64 
 7   air_val_mo  39926 non-null  int64 
 8   air_swt_mo  39926 non-null  int64 
 9   ves_val_mo  39926 non-null  int64 
 10  ves_swt_mo  39926 non-null  int64 
 11  cnt_val_mo  39926 non-null  int64 
 12  cnt_swt_mo  39926 non-null  int64 
 13  gen_val_yr  39926 non-null  object
 14  con_val_yr  39926 non-null  object
 15  air_val_yr  39926 non-null  object
 16  air_swt_yr  39926 non-null  object
 17  ves_val_yr  39926 non-null  object
 18  ves_swt_yr  39926 non-null  object
 19  cnt_val_yr  39926 non-null  object
 20  cnt_swt_yr  39926 n

In [148]:
df_land_imports_mex_wash_2512 = df_land_imports_mex_v1_2512[
    df_land_imports_mex_v1_2512["commodity"].isin(["845011", "845020"])]
df_land_imports_mex_wash_2512.head()

,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr
849178,845011,2010,CA,2025,12,1807730,1807730,0,0,0,...,0,0,27563248,27563248,0,0,0,0,0,0
849179,845011,2010,CO,2025,12,398827,398827,0,0,0,...,0,0,4922868,4922868,0,0,0,0,0,0
849180,845011,2010,FL,2025,12,1378826,1378826,0,0,0,...,0,0,19121576,19121576,0,0,254100,58590,254100,58590
849181,845011,2010,GA,2025,12,764299,764299,0,0,0,...,0,0,13078900,13078900,0,0,0,0,0,0
849182,845011,2010,IL,2025,12,716040,716040,0,0,0,...,0,0,8086170,8086170,0,0,0,0,0,0


In [149]:
df_land_imports_mex_wash_v1_2512 = df_land_imports_mex_wash_2512.copy()

df_land_imports_mex_wash_v1_2512["distance"] = df_land_imports_mex_wash_v1_2512.apply(
    core.compute_distance,
    args=[coord.mexico_start, coord.state_centroids, False, None, "state"], axis=1)

df_land_imports_mex_wash_v1_2512["gen_swt_mo"] = (
    df_land_imports_mex_wash_v1_2512["gen_val_mo"] / washing_machine_price_coeff)

df_land_imports_mex_wash_v1_2512["co2"] = df_land_imports_mex_wash_v1_2512.apply(
    core.compute_co2, args=["gen_swt_mo"], axis=1)

df_land_imports_mex_wash_v1_2512.to_csv("data/intermediate/ISTHS6MM_mex_2512.csv")
df_land_imports_mex_wash_v1_2512.info()

<class 'pandas.DataFrame'>
Index: 29 entries, 849178 to 849386
Data columns (total 24 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   commodity   29 non-null     object 
 1   cty_code    29 non-null     object 
 2   state       29 non-null     object 
 3   year        29 non-null     object 
 4   month       29 non-null     object 
 5   gen_val_mo  29 non-null     int64  
 6   con_val_mo  29 non-null     int64  
 7   air_val_mo  29 non-null     int64  
 8   air_swt_mo  29 non-null     int64  
 9   ves_val_mo  29 non-null     int64  
 10  ves_swt_mo  29 non-null     int64  
 11  cnt_val_mo  29 non-null     int64  
 12  cnt_swt_mo  29 non-null     int64  
 13  gen_val_yr  29 non-null     object 
 14  con_val_yr  29 non-null     object 
 15  air_val_yr  29 non-null     object 
 16  air_swt_yr  29 non-null     object 
 17  ves_val_yr  29 non-null     object 
 18  ves_swt_yr  29 non-null     object 
 19  cnt_val_yr  29 non-null     object 
 2

## Land-based Distance/Weight Data (Mexico, Control — HS 8528 TVs, Dec 2024)

In [150]:
df_land_imports_mex_tv_2412 = df_land_imports_mex_v1_2412[
    df_land_imports_mex_v1_2412["commodity"].str.startswith("8528")]
df_land_imports_mex_tv_2412.head()

,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr
942710,852842,2010,PR,2024,12,0,0,0,0,0,...,0,0,136900,136900,136900,1350,0,0,0,0
942737,852849,2010,PR,2024,12,0,0,0,0,0,...,0,0,162288,162288,162288,1972,0,0,0,0
942847,852852,2010,AL,2024,12,0,0,0,0,0,...,0,0,2290,2290,0,0,0,0,0,0
942848,852852,2010,AZ,2024,12,0,0,0,0,0,...,0,0,16250,16250,14000,297,0,0,0,0
942849,852852,2010,CA,2024,12,3059946,3059946,0,0,0,...,0,0,60472486,60472486,551557,1770,0,0,0,0


In [151]:
df_land_imports_mex_tv_2412[df_land_imports_mex_tv_2412["state"]=="VI"]

,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr
945083,852872,2010,VI,2024,12,0,0,0,0,0,...,0,0,8442,8442,0,0,8442,29,0,0


In [152]:
if not df_asian_tv_2412.empty and df_asian_tv_2412["ves_swt_mo"].sum() > 0:
    tv_price_coeff = df_asian_tv_2412["ves_val_mo"].mean() / df_asian_tv_2412["ves_swt_mo"].mean()
else:
    tv_price_coeff = 15.0  # fallback: ~$15/kg for TVs
print(tv_price_coeff, "$/kg")

17.303979051815883 $/kg


In [153]:
df_land_imports_mex_tv_v1_2412 = df_land_imports_mex_tv_2412.copy()

# I did this, because it is logically impossible for commodities to be imported into the US via the Virgin Islands from Mexico, so this is likely a data error. 
# I will exclude it from the analysis.
df_land_imports_mex_tv_v1_2412 = df_land_imports_mex_tv_v1_2412.drop(df_land_imports_mex_tv_v1_2412[df_land_imports_mex_tv_v1_2412["state"]=="VI"].index, inplace=False)

df_land_imports_mex_tv_v1_2412["distance"] = df_land_imports_mex_tv_v1_2412.apply(
    core.compute_distance,
    args=[coord.mexico_start, coord.state_centroids, False, None, "state"], axis=1)

df_land_imports_mex_tv_v1_2412["gen_swt_mo"] = (
    df_land_imports_mex_tv_v1_2412["gen_val_mo"] / tv_price_coeff)

df_land_imports_mex_tv_v1_2412["co2"] = df_land_imports_mex_tv_v1_2412.apply(
    core.compute_co2, args=["gen_swt_mo"], axis=1)

df_land_imports_mex_tv_v1_2412.to_csv("data/intermediate/ISTHS6MM_mex_tv_2412.csv")
df_land_imports_mex_tv_v1_2412.info()

<class 'pandas.DataFrame'>
Index: 112 entries, 942710 to 945085
Data columns (total 24 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   commodity   112 non-null    object 
 1   cty_code    112 non-null    object 
 2   state       112 non-null    object 
 3   year        112 non-null    object 
 4   month       112 non-null    object 
 5   gen_val_mo  112 non-null    int64  
 6   con_val_mo  112 non-null    int64  
 7   air_val_mo  112 non-null    int64  
 8   air_swt_mo  112 non-null    int64  
 9   ves_val_mo  112 non-null    int64  
 10  ves_swt_mo  112 non-null    int64  
 11  cnt_val_mo  112 non-null    int64  
 12  cnt_swt_mo  112 non-null    int64  
 13  gen_val_yr  112 non-null    object 
 14  con_val_yr  112 non-null    object 
 15  air_val_yr  112 non-null    object 
 16  air_swt_yr  112 non-null    object 
 17  ves_val_yr  112 non-null    object 
 18  ves_swt_yr  112 non-null    object 
 19  cnt_val_yr  112 non-null    object 
 

## Land-based Distance/Weight Data (Mexico, Control — HS 8528 TVs, Dec 2025)

In [154]:
df_land_imports_mex_tv_2512 = df_land_imports_mex_v1_2512[
    df_land_imports_mex_v1_2512["commodity"].str.startswith("8528")]
df_land_imports_mex_tv_2512.head()

,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,cnt_val_mo,cnt_swt_mo,gen_val_yr,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr
992995,852849,2010,CA,2025,12,0,0,0,0,0,...,0,0,28133,28133,0,0,0,0,0,0
993095,852852,2010,AL,2025,12,0,0,0,0,0,...,0,0,20107,20107,0,0,0,0,0,0
993096,852852,2010,AR,2025,12,74797,74797,0,0,0,...,0,0,141945,141945,0,0,0,0,0,0
993097,852852,2010,AZ,2025,12,0,0,0,0,0,...,0,0,38688,38688,0,0,0,0,0,0
993098,852852,2010,CA,2025,12,25343683,25343683,0,0,0,...,0,0,193420909,193420909,119413,279,0,0,0,0


In [155]:
df_land_imports_mex_tv_v1_2512 = df_land_imports_mex_tv_2512.copy()

# I did this, because it is logically impossible for commodities to be imported into the US via the Virgin Islands from Mexico, so this is likely a data error. 
# I will exclude it from the analysis.
df_land_imports_mex_tv_v1_2512 = df_land_imports_mex_tv_v1_2512.drop(df_land_imports_mex_tv_v1_2512[df_land_imports_mex_tv_v1_2512["state"]=="VI"].index, inplace=False)


df_land_imports_mex_tv_v1_2512["distance"] = df_land_imports_mex_tv_v1_2512.apply(
    core.compute_distance,
    args=[coord.mexico_start, coord.state_centroids, False, None, "state"], axis=1)

df_land_imports_mex_tv_v1_2512["gen_swt_mo"] = (
    df_land_imports_mex_tv_v1_2512["gen_val_mo"] / tv_price_coeff)

df_land_imports_mex_tv_v1_2512["co2"] = df_land_imports_mex_tv_v1_2512.apply(
    core.compute_co2, args=["gen_swt_mo"], axis=1)

df_land_imports_mex_tv_v1_2512.to_csv("data/intermediate/ISTHS6MM_mex_tv_2512.csv")
df_land_imports_mex_tv_v1_2512.info()

<class 'pandas.DataFrame'>
Index: 124 entries, 992995 to 995325
Data columns (total 24 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   commodity   124 non-null    object 
 1   cty_code    124 non-null    object 
 2   state       124 non-null    object 
 3   year        124 non-null    object 
 4   month       124 non-null    object 
 5   gen_val_mo  124 non-null    int64  
 6   con_val_mo  124 non-null    int64  
 7   air_val_mo  124 non-null    int64  
 8   air_swt_mo  124 non-null    int64  
 9   ves_val_mo  124 non-null    int64  
 10  ves_swt_mo  124 non-null    int64  
 11  cnt_val_mo  124 non-null    int64  
 12  cnt_swt_mo  124 non-null    int64  
 13  gen_val_yr  124 non-null    object 
 14  con_val_yr  124 non-null    object 
 15  air_val_yr  124 non-null    object 
 16  air_swt_yr  124 non-null    object 
 17  ves_val_yr  124 non-null    object 
 18  ves_swt_yr  124 non-null    object 
 19  cnt_val_yr  124 non-null    object 
 

## Weight Data(USA Trade Online) - irrelevant?

Already well-organized, the weight data does not need much cleaning.

However, the previous datasets(US Census) contains everything necessary. Not just weight of imports, but also origin and destination info. Use this as a sanity check for weight of ocean-side imports.

In [156]:
washing_machine_df_v1 = washing_machine_df.drop("Unnamed: 5", axis=1)
washing_machine_df_v1["Air SWT (kg)"] = washing_machine_df_v1["Air SWT (kg)"].str.replace(',', '')
washing_machine_df_v1["Vessel SWT (kg)"] = washing_machine_df_v1["Vessel SWT (kg)"].str.replace(',', '')

washing_machine_df_v1["Air SWT (kg)"] = pd.to_numeric(washing_machine_df_v1["Air SWT (kg)"])
washing_machine_df_v1["Vessel SWT (kg)"] = pd.to_numeric(washing_machine_df_v1["Vessel SWT (kg)"])

washing_machine_df_v1 = washing_machine_df_v1.fillna(0)

In [157]:
washing_machine_df_v1.head()

,Commodity,Country,Time,Air SWT (kg),Vessel SWT (kg)
0,845011 Washing Mach Automatic W Dry Line Cap N...,China,January 2025,174.0,1777528.0
1,845011 Washing Mach Automatic W Dry Line Cap N...,China,February 2025,0.0,2459827.0
2,845011 Washing Mach Automatic W Dry Line Cap N...,China,March 2025,0.0,2268490.0
3,845011 Washing Mach Automatic W Dry Line Cap N...,China,April 2025,0.0,2512291.0
4,845011 Washing Mach Automatic W Dry Line Cap N...,China,May 2025,0.0,2541012.0


In [158]:
washing_machine_df_v1[washing_machine_df_v1["Country"]=="Mexico"].head()

,Commodity,Country,Time,Air SWT (kg),Vessel SWT (kg)
32,845011 Washing Mach Automatic W Dry Line Cap N...,Mexico,January 2025,0.0,55246.0
33,845011 Washing Mach Automatic W Dry Line Cap N...,Mexico,February 2025,0.0,74445.0
34,845011 Washing Mach Automatic W Dry Line Cap N...,Mexico,March 2025,0.0,239852.0
35,845011 Washing Mach Automatic W Dry Line Cap N...,Mexico,April 2025,0.0,175512.0
36,845011 Washing Mach Automatic W Dry Line Cap N...,Mexico,May 2025,0.0,199598.0


In [159]:
washing_machine_df_v1["Time"].value_counts()

Time
April 2025            9
June 2025             9
July 2025             9
October 2025          9
December 2025         9
2026 through March    9
January 2026          9
January 2025          8
February 2025         8
March 2025            8
May 2025              8
August 2025           8
September 2025        8
November 2025         8
February 2026         8
March 2026            8
Name: count, dtype: int64

In [160]:
washing_machine_df_v1["Air SWT (kg)"].mean()

np.float64(171.28148148148148)

# CO2 Calculations(transportation)

Now that we have the three necessary components - ton-km CO2 factor, weight of imports and distance of imports, then we can calculate the carbon footprint of transportation. This is only one step, as we also need to calculate CO2 of manufacturing(grid intensity x energy spent on manufacturing) in order to understand the full extent of the carbon footprint.

CO2, in this case, is measured in grams. For now, we do this only for 5 countries(China, Vietnam, South Korea, India, Mexico) and only for the month of January 2025, using the datasets taken from census.gov website.

In [161]:
# def compute_co2(row, transportation_type):
#     weight = row[transportation_type]
#     dist = row["distance"]
#     co2_coeff = ton_km[row["cty_code"]]
#     return weight * dist * co2_coeff

df_land_imports_mex_wash_v1_2412["co2"] = df_land_imports_mex_wash_v1_2412.apply(core.compute_co2, args = ["gen_swt_mo"], axis=1)

In [162]:
df_land_imports_mex_wash_v1_2412.head()

,commodity,cty_code,state,year,month,gen_val_mo,con_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,con_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr,distance,gen_swt_mo,co2
799500,845011,2010,CA,2024,12,2805860,2805860,0,0,0,...,28767168,0,0,40635,35100,40635,35100,2847.115733,722529.953219,1.645701e+11
799501,845011,2010,CO,2024,12,430796,430796,0,0,0,...,5790782,0,0,0,0,0,0,2028.894380,110933.194716,1.800574e+10
799502,845011,2010,FL,2024,12,1334984,1334984,0,0,328736,...,19850931,0,0,3583492,831963,3583492,831963,2442.374793,343768.373001,6.716890e+10
799503,845011,2010,GA,2024,12,1199977,1199977,0,0,0,...,16540032,0,0,0,0,0,0,2389.453884,309003.059908,5.906788e+10
799504,845011,2010,IN,2024,12,1157949,1157949,0,0,0,...,13289441,0,0,0,0,0,0,2660.910286,298180.535308,6.347453e+10


In [163]:
df_land_imports_mex_wash_v1_2412.info()

<class 'pandas.DataFrame'>
Index: 32 entries, 799500 to 799719
Data columns (total 24 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   commodity   32 non-null     object 
 1   cty_code    32 non-null     object 
 2   state       32 non-null     object 
 3   year        32 non-null     object 
 4   month       32 non-null     object 
 5   gen_val_mo  32 non-null     int64  
 6   con_val_mo  32 non-null     int64  
 7   air_val_mo  32 non-null     int64  
 8   air_swt_mo  32 non-null     int64  
 9   ves_val_mo  32 non-null     int64  
 10  ves_swt_mo  32 non-null     int64  
 11  cnt_val_mo  32 non-null     int64  
 12  cnt_swt_mo  32 non-null     int64  
 13  gen_val_yr  32 non-null     object 
 14  con_val_yr  32 non-null     object 
 15  air_val_yr  32 non-null     object 
 16  air_swt_yr  32 non-null     object 
 17  ves_val_yr  32 non-null     object 
 18  ves_swt_yr  32 non-null     object 
 19  cnt_val_yr  32 non-null     object 
 2

In [164]:
# df_asian_routes_wash["co2"] = df_asian_routes_wash["vessel_swt_mo"]*df_asian_routes_wash["distance"]*ton_km[df_asian_routes_wash["cty_code"]]
df_asian_tv_2412["co2"] = df_asian_tv_2412.apply(core.compute_co2, args = ["ves_swt_mo"], axis=1)


In [165]:
df_asian_tv_2412.head()

,commodity,cty_code,dist_unlade,port_unlade,year,month,gen_val_mo,air_val_mo,air_swt_mo,ves_val_mo,...,gen_val_yr,air_val_yr,air_swt_yr,ves_val_yr,ves_swt_yr,cnt_val_yr,cnt_swt_yr,port_full,distance,co2
953166,852842,5700,10,12,2024,12,0,0,0,0,...,4810,4810,11,0,0,0,0,1012,19771.475786,0.0
953167,852842,5700,27,09,2024,12,0,0,0,0,...,59160,0,0,59160,2535,59160,2535,2709,10674.961095,0.0
953168,852842,5700,31,26,2024,12,0,0,0,0,...,14575,14575,33,0,0,0,0,3126,7874.489458,0.0
953169,852842,5800,34,01,2024,12,0,0,0,0,...,9519,0,0,0,0,0,0,3401,23391.390800,0.0
953202,852849,5330,41,01,2024,12,0,0,0,0,...,15600,15600,19,0,0,0,0,4101,16159.992314,0.0


In [166]:
df_asian_tv_2412.info()

<class 'pandas.DataFrame'>
Index: 687 entries, 953166 to 956000
Data columns (total 23 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   commodity    687 non-null    object 
 1   cty_code     687 non-null    object 
 2   dist_unlade  687 non-null    object 
 3   port_unlade  687 non-null    object 
 4   year         687 non-null    object 
 5   month        687 non-null    object 
 6   gen_val_mo   687 non-null    int64  
 7   air_val_mo   687 non-null    int64  
 8   air_swt_mo   687 non-null    int64  
 9   ves_val_mo   687 non-null    int64  
 10  ves_swt_mo   687 non-null    int64  
 11  cnt_val_mo   687 non-null    int64  
 12  cnt_swt_mo   687 non-null    int64  
 13  gen_val_yr   687 non-null    object 
 14  air_val_yr   687 non-null    object 
 15  air_swt_yr   687 non-null    object 
 16  ves_val_yr   687 non-null    object 
 17  ves_swt_yr   687 non-null    object 
 18  cnt_val_yr   687 non-null    object 
 19  cnt_swt_yr   687

# Statistics

Now we use the difference-in-difference method to evaluate the extent, to which the tariffs have changed the CO2 footprint. We compare the differences in the control group and the treatment group. 

What will be the control group in our case? 

The control group will be "commodity", more specifically TV sets. TVs have mostly the same demand as washing machines, since both are bought together when people move to a new home. However, unlike the washing machines, TV sets don't fall under the Liberation Day tariffs and steel tariffs of 2025, which is due to goods with semiconductors being exempt from tariffs. These factors make TVs a good control group, since they're similar to washing machines in most key regards, except for the thing that we try to measure.

However, TV sets are lighter than washing machines, so CO2 change scales differently. One way to fix this is to normalize CO2 data by calculating CO2/kg of commodity, rather than total CO2. 


First, we must assemble an econometric model.

Let us start with a basic 2x2 regression equation:

$Y_i = \alpha + \beta*ifTariffShock\_c*\delta*PostTariff\_t+\gamma*(ifTariffShock*PostTariff)+\epsilon_i$

+ $Y_i$ represents grams of CO2 emissions of supply chains of imports from a particular country $c$ at a certain time $t$
+ ifTariffShock_c is a boolean that represents if a country has been affected by the tariff shock. 1 if it is, 0 if not.
+ PostTariff_t is a boolean that represents if the observation takes place before or after the tariff. 0 if the date is Dec 2024, 1 if it's Dec 2025
+ $\gamma$ is the difference-in-differences estimator.

For the model, we need the following assumptions:
+ Parallel trends
    + Flaaen, Hortaçsu & Tintelnot (2020) did this by comparing the trends of the treatment group(washing machines) with the control group(refrigerators, dishwashes and other un-tariffed appliances) before the tariffs.
    + We do this because we cannot see what the countries would've done WITHOUT tariffs. However, we CAN test whether these groups moved in parallel before the tariff.
    + Pre-trend plot necessary to demonstrate it?
+ Anticipation
    + Expectations shape economics. If the importers already knew that the tariffs would've happened, this would've influenced their behavior compared to if they didn't know about the tariffs beforehand.
    + Announcements come many weeks before the actual tariff, as evidenced by Freund et al. (2024). 
+ Error correlates over time within a country
    + How do we address that?
+ Seasonality
    + We assume that washing machine demand is affected by seasons. To avoid the error caused by that, we compare the same month of a different year(Dec 2024 and Dec 2025) respectively. 
    + The announcement date for Liberation Day Tariffs was Feb 13th 2025, so this covers the "anticipation" assumption
+ Semiconductor tariff exception for TV sets is valid
    + While the language around the tariff exception for semiconductors doesn't clarify(rewrite? how exactly is the ambiguity problematic?) the status of TV sets, we assume that they fall under that exemption, and that every party in the supply chain of TVs recognizes that.

This code sets up a DiD model and runs an OLS regression combining **both Mexican and Asian data**. Mexican observations are grouped by destination state; Asian observations are grouped by origin country and destination port.

Then it creates a long table, in which each row is a combination of country, date and product(20 rows in total). Each row has total weight of products imported, total CO2 released and weighted average distance. Each of the aforementioned

In [167]:
def _tag(df, commodity, period):
    out = df[["cty_code", "ves_swt_mo", "co2", "distance"]].copy()
    out["commodity"] = commodity
    out["period"] = period
    return out

df_sea_all = pd.concat([
    _tag(df_asian_wash_2412, "washing_machine", "2412"),
    _tag(df_asian_wash_2512, "washing_machine", "2512"),
    _tag(df_asian_tv_2412,   "tv",              "2412"),
    _tag(df_asian_tv_2512,   "tv",              "2512"),
], ignore_index=True)

df_sea_all["dist_x_weight"] = df_sea_all["distance"] * df_sea_all["ves_swt_mo"]

df_sea_agg = df_sea_all.groupby(["cty_code", "commodity", "period"]).agg(
    total_weight=("ves_swt_mo", "sum"),
    total_co2=("co2", "sum"),
    _dist_x_wt=("dist_x_weight", "sum"),
).reset_index()

df_sea_agg["avg_distance"] = df_sea_agg["_dist_x_wt"] / df_sea_agg["total_weight"]
df_sea_agg = df_sea_agg.drop(columns="_dist_x_wt")
df_sea_agg


,cty_code,commodity,period,total_weight,total_co2,avg_distance
0,5330,tv,2412,0,0.000000e+00,NaN
1,5330,tv,2512,622731,6.731633e+10,18016.426993
2,5330,washing_machine,2412,0,0.000000e+00,NaN
3,5330,washing_machine,2512,0,0.000000e+00,NaN
4,5520,tv,2412,7649143,7.611921e+11,14216.196957
5,5520,tv,2512,12756999,1.349466e+12,15111.767147
6,5520,washing_machine,2412,5508028,6.353774e+11,16479.255459
7,5520,washing_machine,2512,3811283,4.207441e+11,15770.622253
8,5700,tv,2412,26753271,1.241387e+12,11600.326070
9,5700,tv,2512,10131663,5.303737e+11,13087.035353


In [168]:
import statsmodels.api as sm

def prep_did(df, treated_flag, post_flag):
    agg = df.groupby("state").agg(
        co2=("co2", "sum"),
        gen_swt_mo=("gen_swt_mo", "sum"),
    ).reset_index()
    agg["co2_intensity"] = agg["co2"] / agg["gen_swt_mo"]
    agg["treated"] = treated_flag
    agg["post"] = post_flag
    return agg

# Drop zero-weight rows: co2_intensity = 0/0 = NaN for country-commodity-period
# cells with no shipments; undefined intensity should not enter the regression.
df_sea_did = df_sea_agg[df_sea_agg["total_weight"] > 0].copy()
df_sea_did["co2_intensity"] = df_sea_did["total_co2"] / df_sea_did["total_weight"]
df_sea_did["treated"] = (df_sea_did["commodity"] == "washing_machine").astype(int)
df_sea_did["post"]    = (df_sea_did["period"]    == "2512").astype(int)

land_parts = [
    prep_did(df_land_imports_mex_wash_v1_2412, treated_flag=1, post_flag=0),
    prep_did(df_land_imports_mex_wash_v1_2512, treated_flag=1, post_flag=1),
    prep_did(df_land_imports_mex_tv_v1_2412,   treated_flag=0, post_flag=0),
    prep_did(df_land_imports_mex_tv_v1_2512,   treated_flag=0, post_flag=1),
]

df_did = pd.concat([df_sea_did] + land_parts, ignore_index=True)
df_did["treated_x_post"] = df_did["treated"] * df_did["post"]
df_did.sort_values("co2_intensity", ascending=False).head(10)

,cty_code,commodity,period,total_weight,total_co2,avg_distance,co2_intensity,treated,post,state,co2,gen_swt_mo,treated_x_post
31,NaN,NaN,NaN,NaN,NaN,NaN,370700.619623,1,0,PR,2.708836e+10,73073.414171,0
86,NaN,NaN,NaN,NaN,NaN,NaN,370700.619623,0,0,PR,5.072332e+09,13683.095622,0
130,NaN,NaN,NaN,NaN,NaN,NaN,370700.619623,0,1,PR,3.934950e+09,10614.899582,0
48,NaN,NaN,NaN,NaN,NaN,NaN,370700.619623,1,1,PR,3.827644e+10,103254.321499,1
121,NaN,NaN,NaN,NaN,NaN,NaN,338030.574568,0,1,NH,1.232453e+09,3645.982222,0
67,NaN,NaN,NaN,NaN,NaN,NaN,335381.687748,0,0,MA,4.437204e+09,13230.309590,0
113,NaN,NaN,NaN,NaN,NaN,NaN,335381.687748,0,1,MA,9.495263e+09,28311.811898,0
51,NaN,NaN,NaN,NaN,NaN,NaN,316976.735658,1,1,WA,1.507347e+11,475538.558428,1
137,NaN,NaN,NaN,NaN,NaN,NaN,316976.735658,0,1,WA,3.089677e+10,97473.303392,0
94,NaN,NaN,NaN,NaN,NaN,NaN,316976.735658,0,0,WA,2.123613e+10,66995.862427,0


It makes sense for carbon footprint of imports to California, Georgia and Texas to be high, since those states border Mexico. However, why do imports of TVs from Mexico to New Jersey have such high total value and CO2 footprint?

In [169]:
X = sm.add_constant(df_did[["treated", "post", "treated_x_post"]])
model = sm.OLS(df_did["co2_intensity"], X).fit(cov_type="HC3")
print(model.summary())

ValueError: r_matrix performs f_test for using dimensions that are asymptotically non-normal

Having run the OLS regression on the difference-in-differences model, we found that for Mexico's imports, there is no difference in the CO2 before and after the tariffs. 

Why could that be? I know that for land transportation to the USA, the number of trade routes is more limited compared to ocean routes, so there is little that Mexico can do to change their trade routes in the aftermath of tariffs. But how much did the number of imported goods change?

Also, how do I interpret those other statistics? F-statistic, log-likelihood, AIC/BIC, Durbin-Watson, et cetera.